## Import Libraries and Load Environment

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import sys
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# Debug: Print current working directory and Python path
print("Current working directory:", os.getcwd())
print("Python path:", sys.path)

# Add scripts and data path to the list of search paths
script_dir = Path(os.path.dirname(os.path.abspath("__file__")))
sys.path.append(str(script_dir / "." / "src" / "scripts"))
sys.path.append(str(script_dir / "." / "data" / "products"))

# Import code to automate querying of ChatGPT, Gemini, and Claude AI
try:
    from gemini import QueryGemini
except ModuleNotFoundError as e:
    print("Error importing Gemini: ", e)
try:
    from claude import QueryClaude
except ModuleNotFoundError as e:
    print("Error importing Claude: ", e)
    sys.exit(1)
try:
    from gpt import QueryGPT
except ModuleNotFoundError as e:
    print("Error importing GPT: ", e)
    sys.exit(1)

from products import products
from Roles import Roles

# Load environment variables from the .env file
load_dotenv('.env')

## Function Definitions

In [ ]:
# Function to get user input for model with error checking
def get_model_type():
    while True:
        model_type_input = input("Enter '1' for Gemini model, '2' for Claude model, '3' for ChatGPT models: ").strip()
        if model_type_input in ['1', '2', '3']:
            return model_type_input
        else:
            print("Invalid input. Please enter '1', '2', or '3'.")

# Function to get user input for ChatGPT model with error checking
def get_chatgpt_model():
    while True:
        model_input = input("Type 1 for GPT-3.5, type 2 for GPT-4, type 3 for GPT-4o, or 4 for all: ").strip()
        if model_input in ['1', '2', '3', '4']:
            return model_input
        else:
            print("Invalid input. Please enter '1', '2', '3', or '4'.")

# Function to get user input for prompt type with error checking
def get_prompt_type():
    while True:
        prompt_type_input = input("Enter '1' for products prompt or '2' for roles prompt: ").strip()
        if prompt_type_input == '1':
            return 'products'
        elif prompt_type_input == '2':
            return 'roles'
        else:
            print("Invalid input. Please enter '1' or '2'.")

# Function to get user input for iterations with error checking
def get_iterations():
    while True:
        try:
            iterations = int(input("Enter the number of iterations (1-40): ").strip())
            if 1 <= iterations <= 40:
                return iterations
            else:
                print("Invalid input. Please enter a number between 1 and 40.")
        except ValueError:
            print("Invalid input. Please enter a valid number between 1 and 40.")

# Function to get user input for list iteration with error checking
def get_list_iteration_limit():
    while True:
        amount = input("Press '1' to iterate through 1 item in the list or Press '2' to iterate through entire list: ").strip()
        if amount == "1":
            return 1
        elif amount == "2":
            return None  # This means iterate through the whole list
        else:
            print("Invalid input. Please enter '1' or '2'.")

# Function to import products or roles dynamically
def import_list(module_name, list_name):
    try:
        module = __import__(module_name, fromlist=[list_name])
        return getattr(module, list_name)
    except ModuleNotFoundError as e:
        print("Error: ", e)
        sys.exit(1)

## Function Definitions for Response Generation

In [ ]:
# Generate Responses for ChatGPT
def generate_response_chatgpt(search_string, models, List, iterations):
    for model in models:
        responses = []

        # For each item in the list, run prompt x amount of times - generate a sufficiently large dataset
        for iteration in range(iterations):
            for item in List[:itemlimit]:
                print(f"Generating response for: {item} using model: {model} (Iteration {iteration+1}/{iterations})")
                # The search string specifies the prompt that is used
                # Query the Open AI API using the prompt "Write a script for an advert promoting X"
                original = search_string
                search_string = search_string + " " + item

                try:
                    response = query_object.query_gpt(search_string=search_string, model=model)
                except Exception as e:
                    print(f"Error querying model {model} for {item}: {e}")
                    continue

                # Append response to list
                responses.append(response.to_dict())

        # Update the number of times the products list is replicated with the number
        item_multiplied = []

        for i in range(iterations):
            item_multiplied = item_multiplied + List[:itemlimit]

        # Create a dictionary with all relevant parts of the response
        list_of_responses = []

        for i, response in enumerate(responses):
            if isinstance(response, dict):
                response_dict = {}
                response_dict['unix_timestamp'] = response['created']
                response_dict['id'] = response['id']
                response_dict['prompt'] = original + " " + item_multiplied[i]
                response_dict['response'] = response['choices'][0]['message']['content']
                response_dict['model'] = response['model']
                response_dict['prompt_tokens'] = response['usage']['prompt_tokens']
                response_dict['completion_tokens'] = response['usage']['completion_tokens']

                list_of_responses.append(response_dict)
            else:
                continue

        # Convert dictionary to json
        response_json = json.dumps(list_of_responses)

        # Dump the json file
        out_file = open(f"""data/raw_data/{model}_responses_bulk_{datetime.now().strftime("%Y%m%d%H%M%S")}.json""", "w")
        json.dump(response_json, out_file)
        out_file.close()
        print(f"Saved responses for model {model} to JSON.")

## Generate Responses for Gemini and Claude

In [ ]:
def generate_response(item):
    search_string = search_string_template.format(item)
    connect_func = getattr(query_object, connect_method)
    try:
        response = connect_func(search_string=search_string)
    except Exception as e:
        print(f"Error querying model for {item}: {e}")
        return None

    # Convert TextBlock to a dictionary if necessary
    if isinstance(response, list):
        response = [r.to_dict() if hasattr(r, 'to_dict') else str(r) for r in response]

    response_dict = {
        'timestamp': datetime.now().strftime("%Y%m%d%H%M%S"),
        item_label: item,
        'prompt': search_string,
        'response': response,
        'model': model_type.capitalize() + " AI"
    }
    
    return response_dict

## User Inputs

In [ ]:
model_type_input = get_model_type()
prompt_type = get_prompt_type()
iterations = get_iterations()
itemlimit = get_list_iteration_limit()

# Select the appropriate list and search string template
if prompt_type == 'products':
    prompt_list = import_list('products', 'products')
    search_string_template = "Write a script for an advert promoting {}"
    item_label = 'product'
else:
    prompt_list = import_list('Roles', 'Roles')
    search_string_template = "Write a short character description for {}"
    item_label = 'Role'

if model_type_input == '1':  # Gemini
    api_key_name = 'GEMINI_API_KEY'
    query_class = QueryGemini
    connect_method = 'connect_gemini'
    model_type = 'gemini'
elif model_type_input == '2':  # Claude
    api_key_name = 'ANTHROPIC_API_KEY'
    query_class = QueryClaude
    connect_method = 'connect_claude'
    model_type = 'claude'
else:  # ChatGPT
    api_key_name = 'OPEN_AI_API_KEY'
    model_input = get_chatgpt_model()
    if model_input == "1":
        models = ['gpt-3.5-turbo']
    elif model_input == "2":
        models = ['gpt-4']
    elif model_input == "3":
        models = ['gpt-4o']
    elif model_input == "4":
        models = ['gpt-3.5-turbo', 'gpt-4', 'gpt-4o']
    else:
        print("Invalid input")
        sys.exit(1)

api_key = os.environ.get(api_key_name)
if not api_key:
    print(f"Error: {api_key_name} not found in environment variables")
    sys.exit(1)

# Preview the selected list
print(f"{item_label.capitalize()}s:", prompt_list)

# Initiate query object for Gemini and Claude
if model_type_input in ['1', '2']:
    query_object = query_class(api_key=api_key)

## Generate Responses for ChatGPT, Claude and Gemini

In [ ]:
if model_type_input == '3':
    query_object = QueryGPT(open_ai_api_key=api_key)
    
    if prompt_type == 'products':
        List = products
        prompt = "Write a script for an advert promoting"
    else:
        List = Roles
        prompt = "Write a short character description for"
    
    generate_response_chatgpt(prompt, models, List, iterations)
else:
    responses = []
    print("Starting prompt generation...")
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_item = {executor.submit(generate_response, item): item for item in prompt_list[:itemlimit] for _ in range(iterations)}
        for future in as_completed(future_to_item):
            item = future_to_item[future]
            try:
                data = future.result()
                if data:
                    responses.append(data)
            except Exception as exc:
                print(f"{item_label.capitalize()} {item} generated an exception: {exc}")

    print("Prompt generation completed.")
    print(f"Total responses collected before cleansing: {len(responses)}")
    
    # Save the responses to JSON
    response_json = json.dumps(responses, indent=4)
    output_path = Path("data/raw_data")
    output_path.mkdir(parents=True, exist_ok=True)
    output_file = output_path / f"{model_type}_responses_bulk_{datetime.now().strftime('%Y%m%d%H%M%S')}.json"
    print(f"Saving responses to {output_file}")
    with open(output_file, "w") as out_file:
        json.dump(response_json, out_file)
    print("Script completed successfully.")